# 2.1 — Calidad de datos

## Unidad 2: Caracteristicas de Productos de Datos

En la Unidad 1 limpiamos datos a mano: encontramos nulos, los arreglamos, seguimos adelante. Eso funciona una vez. Pero un producto de datos recibe datos nuevos cada dia, cada semana, cada hora. No podemos sentarnos a revisar cada carga.

La calidad de datos no es limpiar — es escribir reglas que validan automaticamente. Si los datos cumplen las reglas, pasan. Si no, el pipeline se detiene antes de que el dato sucio llegue al dashboard.

### Contenido:
1. La diferencia entre limpiar y validar
2. Validacion con pandera — esquemas sobre DataFrames
3. Validacion con Pydantic — esquemas sobre registros
4. Validacion con Great Expectations — suites de pruebas
5. Integrar validacion en un pipeline

In [ ]:
# ============================================================
# INSTALACION
# ============================================================

# !pip install pandera pydantic great_expectations

In [ ]:
# ============================================================
# DATOS DE EJEMPLO
# ============================================================

# Simulamos datos que llegan a un pipeline
# Algunos son correctos, otros tienen errores

import pandas as pd
import numpy as np

# Datos "buenos" — lo que esperamos recibir
datos_buenos = pd.DataFrame({
    "fecha": pd.to_datetime(["2024-06-01", "2024-06-02", "2024-06-03", "2024-06-04"]),
    "producto": ["dashboard", "reporte", "api", "app web"],
    "region": ["bogota", "medellin", "cali", "manizales"],
    "unidades": [15, 8, 22, 5],
    "precio": [450.0, 280.0, 620.0, 350.0],
    "calificacion": [4, 5, 3, 4],
})

# Datos "malos" — lo que puede llegar en la vida real
datos_malos = pd.DataFrame({
    "fecha": pd.to_datetime(["2024-06-01", "2024-06-02", "2024-06-03", "2024-06-04", "2024-06-05"]),
    "producto": ["dashboard", "REPORTE", "api", "producto_nuevo", None],
    "region": ["bogota", "medellin", "tokio", "cali", "manizales"],
    "unidades": [15, -3, 22, 5000, 8],
    "precio": [450.0, 280.0, -10.0, 620.0, None],
    "calificacion": [4, 7, 3, 4, 0],
})

print("Datos buenos:")
print(datos_buenos.to_string(index=False))
print("\nDatos malos:")
print(datos_malos.to_string(index=False))

---
## 1. La diferencia entre limpiar y validar

| | Limpiar (Unidad 1) | Validar (Unidad 2) |
|---|---|---|
| Cuando | Despues de que los datos llegaron | Antes de que entren al sistema |
| Quien | Un analista, una vez | Codigo automatico, siempre |
| Resultado | Datos corregidos | Datos aprobados o rechazados |
| Analogia | Limpiar el cuarto | Poner reglas para que no se ensucie |

La validacion responde una pregunta simple: estos datos cumplen las reglas que definimos? Si la respuesta es no, no los dejamos pasar.

---
## 2. Validacion con pandera

pandera valida DataFrames de pandas contra un esquema. El esquema define que columnas debe tener, que tipo, que rango, que valores son aceptables. Si el DataFrame no cumple, pandera lanza un error con el detalle de que fallo.

### 2.1 Esquema basico

In [ ]:
# ============================================================
# PANDERA — DEFINIR UN ESQUEMA
# ============================================================

import pandera.pandas as pa

# Un esquema define las reglas para cada columna
# pa.Column(tipo, checks=[lista de validaciones])

esquema_ventas = pa.DataFrameSchema({
    
    # La fecha debe ser datetime, no puede ser nula
    "fecha": pa.Column(
        "datetime64[ns]",
        nullable=False
    ),
    
    # El producto debe ser uno de los 5 permitidos
    "producto": pa.Column(
        str,
        checks=pa.Check.isin(["dashboard", "reporte", "api", "app web", "pipeline etl"]),
        nullable=False
    ),
    
    # La region debe ser una de las 5 ciudades
    "region": pa.Column(
        str,
        checks=pa.Check.isin(["bogota", "medellin", "cali", "manizales", "barranquilla"]),
        nullable=False
    ),
    
    # Unidades: entero positivo, maximo razonable 200
    "unidades": pa.Column(
        int,
        checks=[
            pa.Check.greater_than(0),
            pa.Check.less_than_or_equal_to(200)
        ],
        nullable=False
    ),
    
    # Precio: float positivo
    "precio": pa.Column(
        float,
        checks=pa.Check.greater_than(0),
        nullable=False
    ),
    
    # Calificacion: entero entre 1 y 5
    "calificacion": pa.Column(
        int,
        checks=pa.Check.in_range(1, 5),
        nullable=False
    ),
})

print("Esquema definido con 6 columnas y sus reglas.")

In [ ]:
# ============================================================
# VALIDAR DATOS BUENOS — deben pasar
# ============================================================

# .validate() retorna el DataFrame si pasa, lanza error si no
try:
    df_validado = esquema_ventas.validate(datos_buenos)
    print("PASO: los datos buenos cumplen el esquema.")
    print(f"Filas validadas: {len(df_validado)}")
except pa.errors.SchemaError as e:
    print(f"FALLO: {e}")

In [ ]:
# ============================================================
# VALIDAR DATOS MALOS — deben fallar
# ============================================================

# Con los datos malos, pandera debe detectar los errores
try:
    df_validado = esquema_ventas.validate(datos_malos)
    print("PASO: (esto no deberia pasar)")
except pa.errors.SchemaError as e:
    print("FALLO (esperado). Errores detectados:")
    print()
    # El error tiene un DataFrame con el detalle
    print(e)

In [ ]:
# ============================================================
# MODO LAZY — recoger TODOS los errores en vez de parar en el primero
# ============================================================

# Por defecto pandera para en el primer error
# lazy=True recopila todos los errores y los reporta juntos

try:
    esquema_ventas.validate(datos_malos, lazy=True)
except pa.errors.SchemaErrors as e:
    print(f"Total de errores: {len(e.failure_cases)}")
    print()
    # e.failure_cases es un DataFrame con todos los errores
    print(e.failure_cases[["schema_context", "column", "check", "failure_case"]].to_string(index=False))

### 2.2 Checks personalizados

In [ ]:
# ============================================================
# CHECKS PERSONALIZADOS — logica propia
# ============================================================

# Pa.Check recibe una funcion que retorna True/False por elemento
# o una funcion que recibe la Serie completa y retorna True/False

esquema_avanzado = pa.DataFrameSchema({
    
    "producto": pa.Column(
        str,
        checks=[
            # Check personalizado: debe estar en minusculas
            pa.Check(
                lambda s: s == s.str.lower(),
                error="El producto debe estar en minusculas"
            ),
            # Check personalizado: no debe tener espacios al inicio o final
            pa.Check(
                lambda s: s == s.str.strip(),
                error="El producto no debe tener espacios al inicio o final"
            ),
        ]
    ),
    
    "precio": pa.Column(
        float,
        checks=[
            pa.Check.greater_than(0),
            # Check a nivel de Serie: el promedio debe ser mayor a 100
            pa.Check(
                lambda s: s.mean() > 100,
                error="El precio promedio del lote es sospechosamente bajo"
            ),
        ]
    ),
})

print("Esquema con checks personalizados definido.")

### 2.3 Esquema con decoradores (forma avanzada)

In [ ]:
# ============================================================
# PANDERA CON CLASES — SchemaModel
# ============================================================

# Forma mas limpia de definir esquemas, similar a Pydantic
# Cada campo es un atributo de la clase

import pandera as pa
from pandera.typing import Series

class EsquemaVentas(pa.DataFrameModel):
    """Esquema de validacion para el DataFrame de ventas."""
    
    fecha: Series[pa.DateTime] = pa.Field(nullable=False)
    producto: Series[str] = pa.Field(
        isin=["dashboard", "reporte", "api", "app web", "pipeline etl"],
        nullable=False
    )
    region: Series[str] = pa.Field(
        isin=["bogota", "medellin", "cali", "manizales", "barranquilla"],
        nullable=False
    )
    unidades: Series[int] = pa.Field(gt=0, le=200, nullable=False)
    precio: Series[float] = pa.Field(gt=0, nullable=False)
    calificacion: Series[int] = pa.Field(ge=1, le=5, nullable=False)
    
    class Config:
        strict = True  # No permite columnas adicionales
        coerce = False # No intenta convertir tipos automaticamente

# Validar con la clase
try:
    EsquemaVentas.validate(datos_buenos)
    print("PASO con SchemaModel")
except pa.errors.SchemaError as e:
    print(f"FALLO: {e}")

---
## 3. Validacion con Pydantic

Pydantic valida registros individuales (diccionarios, JSON). Es el estandar en Python para validar datos que vienen de APIs, archivos de configuracion o formularios. Mientras pandera valida DataFrames completos, Pydantic valida fila por fila.

In [ ]:
# ============================================================
# PYDANTIC — DEFINIR UN MODELO DE DATOS
# ============================================================

from pydantic import BaseModel, Field, field_validator
from datetime import date
from typing import Literal

class TransaccionVenta(BaseModel):
    """Modelo de validacion para una transaccion de venta."""
    
    # Literal restringe a valores exactos
    producto: Literal["dashboard", "reporte", "api", "app web", "pipeline etl"]
    region: Literal["bogota", "medellin", "cali", "manizales", "barranquilla"]
    
    # Field define restricciones numericas
    unidades: int = Field(gt=0, le=200, description="Unidades vendidas")
    precio: float = Field(gt=0, description="Precio unitario en COP")
    calificacion: int = Field(ge=1, le=5, description="Calificacion del cliente")
    
    fecha: date
    
    # Validador personalizado: producto debe estar en minusculas
    @field_validator("producto")
    @classmethod
    def producto_minusculas(cls, v):
        if v != v.lower():
            raise ValueError(f"El producto debe estar en minusculas, recibido: '{v}'")
        return v

print("Modelo Pydantic definido.")
print(f"Campos: {list(TransaccionVenta.model_fields.keys())}")

In [ ]:
# ============================================================
# VALIDAR UN REGISTRO BUENO
# ============================================================

# Pydantic recibe un diccionario y retorna un objeto validado
registro_bueno = {
    "producto": "dashboard",
    "region": "bogota",
    "unidades": 15,
    "precio": 450.0,
    "calificacion": 4,
    "fecha": "2024-06-01"
}

venta = TransaccionVenta(**registro_bueno)
print("PASO:")
print(f"  Producto: {venta.producto}")
print(f"  Region: {venta.region}")
print(f"  Ingreso: {venta.unidades * venta.precio}")

In [ ]:
# ============================================================
# VALIDAR UN REGISTRO MALO
# ============================================================

from pydantic import ValidationError

registro_malo = {
    "producto": "DASHBOARD",        # Mayusculas
    "region": "tokio",              # No esta en la lista
    "unidades": -3,                 # Negativo
    "precio": 0,                    # Cero
    "calificacion": 7,              # Fuera de rango
    "fecha": "2024-06-01"
}

try:
    venta = TransaccionVenta(**registro_malo)
except ValidationError as e:
    print(f"FALLO (esperado). Errores: {e.error_count()}")
    print()
    for error in e.errors():
        campo = error['loc'][0]
        mensaje = error['msg']
        print(f"  {campo}: {mensaje}")

In [ ]:
# ============================================================
# VALIDAR UN DATAFRAME COMPLETO FILA POR FILA
# ============================================================

# Convertir DataFrame a lista de diccionarios y validar cada uno
# Esto permite saber exactamente cuales filas fallaron

def validar_dataframe(df, modelo):
    """
    Valida cada fila de un DataFrame contra un modelo Pydantic.
    Retorna dos DataFrames: validas y rechazadas.
    """
    validas = []
    rechazadas = []
    
    for i, fila in df.iterrows():
        try:
            registro = modelo(**fila.to_dict())
            validas.append(fila)
        except ValidationError as e:
            errores = "; ".join(
                f"{err['loc'][0]}: {err['msg']}" for err in e.errors()
            )
            fila_con_error = fila.copy()
            fila_con_error["errores"] = errores
            rechazadas.append(fila_con_error)
    
    df_validas = pd.DataFrame(validas) if validas else pd.DataFrame()
    df_rechazadas = pd.DataFrame(rechazadas) if rechazadas else pd.DataFrame()
    
    return df_validas, df_rechazadas

# Aplicar a los datos malos
validas, rechazadas = validar_dataframe(datos_malos, TransaccionVenta)

print(f"Filas validas: {len(validas)}")
print(f"Filas rechazadas: {len(rechazadas)}")

if len(rechazadas) > 0:
    print("\nDetalle de rechazadas:")
    print(rechazadas[["producto", "region", "unidades", "precio", "errores"]].to_string(index=False))

### Pandera vs Pydantic — cuando usar cada uno

| | pandera | Pydantic |
|---|---|---|
| Valida | DataFrames completos | Registros individuales |
| Mejor para | Archivos CSV, tablas SQL, lotes | APIs, JSON, formularios |
| Checks | Estadisticos (media, distribucion) | Por campo (tipo, rango) |
| Ecosistema | pandas, pyspark | FastAPI, Django, cualquier backend |

---
## 4. Validacion con Great Expectations

Great Expectations (GX) es el framework mas completo para calidad de datos. Define "expectativas" sobre los datos (como pruebas unitarias pero para datos), las ejecuta y genera reportes HTML con los resultados.

In [ ]:
# ============================================================
# GREAT EXPECTATIONS — SETUP BASICO
# ============================================================

import great_expectations as gx

# Crear un contexto efimero (sin archivos de config)
context = gx.get_context()

# Conectar nuestro DataFrame como fuente de datos
data_source = context.data_sources.add_pandas("ventas")
data_asset = data_source.add_dataframe_asset(name="transacciones")
batch_definition = data_asset.add_batch_definition_whole_dataframe("lote_completo")

print("Contexto de Great Expectations creado.")

In [ ]:
# ============================================================
# DEFINIR EXPECTATIVAS
# ============================================================

# Crear una suite de expectativas (como una suite de tests)
suite = context.suites.add(
    gx.ExpectationSuite(name="validacion_ventas")
)

# Cada expectativa es una regla sobre los datos
# El nombre del metodo describe lo que espera

# 1. La columna 'producto' debe existir
suite.add_expectation(
    gx.expectations.ExpectColumnToExist(column="producto")
)

# 2. La columna 'producto' no debe tener nulos
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="producto")
)

# 3. Los valores de 'producto' deben ser de un conjunto
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="producto",
        value_set=["dashboard", "reporte", "api", "app web", "pipeline etl"]
    )
)

# 4. 'unidades' debe ser positivo
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="unidades",
        min_value=1,
        max_value=200
    )
)

# 5. 'precio' no debe tener nulos
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column="precio")
)

# 6. 'precio' debe ser positivo
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="precio",
        min_value=0.01
    )
)

# 7. 'calificacion' entre 1 y 5
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="calificacion",
        min_value=1,
        max_value=5
    )
)

# 8. No debe haber filas duplicadas
suite.add_expectation(
    gx.expectations.ExpectCompoundColumnsToBeUnique(
        column_list=["fecha", "producto", "region"]
    )
)

# 9. La tabla debe tener entre 1 y 10000 filas
suite.add_expectation(
    gx.expectations.ExpectTableRowCountToBeBetween(
        min_value=1,
        max_value=10000
    )
)

print(f"Suite definida con {len(suite.expectations)} expectativas.")

In [ ]:
# ============================================================
# EJECUTAR VALIDACION SOBRE DATOS BUENOS
# ============================================================

# Crear un validation definition
validation_def = context.validation_definitions.add(
    gx.ValidationDefinition(
        name="validar_datos_buenos",
        data=batch_definition,
        suite=suite
    )
)

# Ejecutar
batch_params = {"dataframe": datos_buenos}
result = validation_def.run(batch_parameters=batch_params)

print(f"Resultado: {'PASO' if result.success else 'FALLO'}")
print(f"Expectativas evaluadas: {len(result.results)}")
for r in result.results:
    status = 'OK' if r.success else 'FALLO'
    exp_type = type(r.expectation_config).__name__
    print(f"  [{status}] {exp_type}")

In [ ]:
# ============================================================
# EJECUTAR VALIDACION SOBRE DATOS MALOS
# ============================================================

validation_def2 = context.validation_definitions.add(
    gx.ValidationDefinition(
        name="validar_datos_malos",
        data=batch_definition,
        suite=suite
    )
)

batch_params = {"dataframe": datos_malos}
result_malo = validation_def2.run(batch_parameters=batch_params)

print(f"Resultado: {'PASO' if result_malo.success else 'FALLO'}")
print()

for r in result_malo.results:
    status = 'OK' if r.success else 'FALLO'
    exp_type = type(r.expectation_config).__name__
    print(f"  [{status}] {exp_type}")
    if not r.success and hasattr(r.result, 'get'):
        unexpected = r.result.get('unexpected_count', 0)
        if unexpected:
            print(f"          {unexpected} valores inesperados")

---
## 5. Integrar validacion en un pipeline

Todo lo anterior fue interactivo — en un notebook, mirando los resultados. En produccion, la validacion es un paso automatico dentro de un pipeline: llegan datos, se validan, si pasan siguen al dashboard, si no se rechazan y alguien recibe una alerta.

In [ ]:
# ============================================================
# FUNCION DE PIPELINE CON VALIDACION
# ============================================================

from datetime import datetime
import json

def pipeline_ventas(ruta_csv, esquema):
    """
    Pipeline que carga, valida y procesa datos de ventas.
    Si la validacion falla, rechaza el lote completo.
    
    Parametros:
        ruta_csv (str): ruta al archivo CSV
        esquema: esquema de pandera para validar
    
    Retorna:
        dict: resultado del pipeline con metricas
    """
    timestamp = datetime.now().isoformat()
    resultado = {
        "timestamp": timestamp,
        "archivo": ruta_csv,
        "estado": None,
        "filas": 0,
        "errores": [],
    }
    
    # 1. Cargar
    try:
        df = pd.read_csv(ruta_csv, parse_dates=["fecha"])
        resultado["filas"] = len(df)
        print(f"[{timestamp}] Cargado: {len(df)} filas")
    except Exception as e:
        resultado["estado"] = "ERROR_CARGA"
        resultado["errores"].append(str(e))
        print(f"[{timestamp}] Error de carga: {e}")
        return resultado
    
    # 2. Validar
    try:
        df_validado = esquema.validate(df, lazy=True)
        resultado["estado"] = "APROBADO"
        print(f"[{timestamp}] Validacion: APROBADO")
    except pa.errors.SchemaErrors as e:
        resultado["estado"] = "RECHAZADO"
        resultado["errores"] = e.failure_cases.to_dict("records")
        print(f"[{timestamp}] Validacion: RECHAZADO")
        print(f"[{timestamp}] Errores: {len(e.failure_cases)}")
        return resultado
    
    # 3. Procesar (solo si paso la validacion)
    df_validado["ingreso"] = df_validado["unidades"] * df_validado["precio"]
    resultado["ingreso_total"] = df_validado["ingreso"].sum()
    print(f"[{timestamp}] Procesado: ingreso total = ${resultado['ingreso_total']:,.0f}")
    
    # 4. Guardar
    df_validado.to_csv("datos_aprobados.csv", index=False)
    print(f"[{timestamp}] Guardado: datos_aprobados.csv")
    
    return resultado

In [ ]:
# ============================================================
# PROBAR CON DATOS BUENOS
# ============================================================

# Guardar datos buenos como CSV para simular la carga
datos_buenos.to_csv("lote_bueno.csv", index=False)

print("=" * 50)
print("  PIPELINE — LOTE BUENO")
print("=" * 50)
resultado_bueno = pipeline_ventas("lote_bueno.csv", esquema_ventas)
print(f"\nEstado final: {resultado_bueno['estado']}")

In [ ]:
# ============================================================
# PROBAR CON DATOS MALOS
# ============================================================

datos_malos.to_csv("lote_malo.csv", index=False)

print("=" * 50)
print("  PIPELINE — LOTE MALO")
print("=" * 50)
resultado_malo = pipeline_ventas("lote_malo.csv", esquema_ventas)
print(f"\nEstado final: {resultado_malo['estado']}")
print(f"Errores: {len(resultado_malo['errores'])}")

In [ ]:
# ============================================================
# GUARDAR LOG DE VALIDACION
# ============================================================

# En produccion, cada ejecucion del pipeline deja un log
# para rastrear que lotes pasaron y cuales no

log = [resultado_bueno, resultado_malo]

with open("log_validacion.json", "w") as f:
    json.dump(log, f, indent=2, default=str)

print("Log guardado: log_validacion.json")
print("Este log se puede cargar en un dashboard de calidad de datos.")

---
## Resumen

| Herramienta | Que valida | Cuando usarla |
|---|---|---|
| **pandera** | DataFrames completos contra un esquema | Archivos CSV, cargas batch, tablas SQL |
| **Pydantic** | Registros individuales (dicts/JSON) | APIs, formularios, datos en streaming |
| **Great Expectations** | Suites de expectativas con reportes | Pipelines de datos en produccion |

### Lo esencial

| Concepto | Lo que importa |
|---|---|
| **Esquema** | Define que columnas, que tipos, que rangos y que valores acepta |
| **Validacion** | Ejecuta el esquema contra los datos y dice si pasan o no |
| **lazy=True** | Recopila todos los errores en vez de parar en el primero |
| **Pipeline** | La validacion es un paso automatico: cargar, validar, procesar o rechazar |
| **Log** | Cada ejecucion deja registro de que paso y que fallo |

### Siguiente paso
En el **Notebook 2.2** veremos como disenar un producto de datos que sea compartible, modular y documentado — las caracteristicas que hacen que otros equipos puedan usarlo sin depender de quien lo construyo.